# Week 12 Lab — Structured Streaming

**CS570 Big Data Processing & Analytics | Spring 2026 | Dr. Ragnar Lesch**

---

**Scenario:** You work at a music streaming platform. Listener play events arrive continuously — thousands per minute. The analytics team needs a real-time dashboard showing trending genres, mood shifts, and artist spikes. Your job: build the streaming pipeline.

**What you will do:**
1. Simulate a live stream of play events from the Spotify catalog
2. Read the stream with Structured Streaming
3. Enrich events by joining with a static catalog
4. Compute running counts, windowed aggregations, and sliding-window mood tracking
5. Handle late-arriving data with watermarks
6. Monitor pipeline health

**Dataset:** 114K Spotify tracks (same dataset from Week 11). We treat it as a static catalog and simulate play events from it.

## Setup

In [ ]:
import os
import shutil
import time
import random
import threading
from datetime import datetime, timedelta

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

spark = (
    SparkSession.builder
    .appName("CS570_Week12_Streaming")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")       # fewer partitions for local mode
    .config("spark.sql.streaming.schemaInference", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print(f"Spark version: {spark.version}")
print(f"Cores: {spark.sparkContext.defaultParallelism}")

KeyboardInterrupt: 

: 

### Load the Static Catalog

This is our reference table — every track in the Spotify dataset with its genre, artist, and audio features. Streaming events carry only `track_id`; we join against this catalog to get the rest.

In [ ]:
from pyspark.sql.types import (StructType, StructField, StringType,
                               IntegerType, DoubleType, BooleanType, LongType)

catalog_schema = StructType([
    StructField("_c0",               IntegerType(),  True),
    StructField("track_id",          StringType(),   True),
    StructField("artists",           StringType(),   True),
    StructField("album_name",        StringType(),   True),
    StructField("track_name",        StringType(),   True),
    StructField("popularity",        IntegerType(),  True),
    StructField("duration_ms",       LongType(),     True),
    StructField("explicit",          BooleanType(),  True),
    StructField("danceability",      DoubleType(),   True),
    StructField("energy",            DoubleType(),   True),
    StructField("key",               IntegerType(),  True),
    StructField("loudness",          DoubleType(),   True),
    StructField("mode",              IntegerType(),  True),
    StructField("speechiness",       DoubleType(),   True),
    StructField("acousticness",      DoubleType(),   True),
    StructField("instrumentalness",  DoubleType(),   True),
    StructField("liveness",          DoubleType(),   True),
    StructField("valence",           DoubleType(),   True),
    StructField("tempo",             DoubleType(),   True),
    StructField("time_signature",    IntegerType(),  True),
    StructField("track_genre",       StringType(),   True),
])

catalog = spark.read.csv("spotify.csv", header=True, schema=catalog_schema)
print(f"Catalog: {catalog.count():,} tracks, {len(catalog.columns)} columns")
catalog.select("track_id", "artists", "track_name", "track_genre",
               "energy", "valence", "danceability", "tempo").show(5, truncate=40)

In [ ]:
# Sanity check: track count per genre — quickly reveals ingestion issues
# (malformed rows land in the wrong genre or get null, showing up here)
catalog.groupBy("track_genre") \
       .count() \
       .orderBy(F.col("count").desc()) \
       .show(100, truncate=False)

## Build the Play-Event Simulator

Real streaming systems ingest data from Kafka, Kinesis, or Pub/Sub. For this lab, we simulate a live stream by writing small CSV files into a watched directory every few seconds. Structured Streaming picks up each new file automatically.

Each "play event" has three fields:
- `track_id` — which song was played
- `user_id` — which listener (random 1–1000)
- `timestamp` — when the play happened (with slight jitter)

We also inject a few **deliberately late events** (timestamp 5–15 minutes in the past) to demonstrate watermarking later.

In [ ]:
# Directory where the simulator writes CSV batches
STREAM_DIR = "play_events"
CHECKPOINT_BASE = "checkpoints"

def clean_dirs():
    """Remove stream and checkpoint directories for a fresh start."""
    for d in [STREAM_DIR, CHECKPOINT_BASE]:
        if os.path.exists(d):
            shutil.rmtree(d)
    os.makedirs(STREAM_DIR, exist_ok=True)
    os.makedirs(CHECKPOINT_BASE, exist_ok=True)
    print(f"Cleaned and created: {STREAM_DIR}/, {CHECKPOINT_BASE}/")

clean_dirs()

In [ ]:
def play_event_simulator(catalog_pdf, stream_dir, stop_event,
                         batch_size=75, interval=4, late_fraction=0.05):
    """
    Background thread that writes CSV batches of play events.

    Parameters
    ----------
    catalog_pdf : pandas DataFrame with at least a 'track_id' column
    stream_dir  : directory to write CSV files into
    stop_event  : threading.Event — set this to stop the simulator
    batch_size  : rows per CSV file (default 75)
    interval    : seconds between batches (default 4)
    late_fraction : fraction of events with timestamps 5-15 min in the past
    """
    track_ids = catalog_pdf["track_id"].tolist()
    batch_num = 0

    while not stop_event.is_set():
        rows = []
        now = datetime.now()

        for _ in range(batch_size):
            tid = random.choice(track_ids)
            uid = random.randint(1, 1000)

            # Most events: current time +/- a few seconds of jitter
            if random.random() < late_fraction:
                # Late event: 5-15 minutes in the past
                ts = now - timedelta(minutes=random.uniform(5, 15))
            else:
                ts = now + timedelta(seconds=random.uniform(-2, 2))

            rows.append(f"{tid},{uid},{ts.strftime('%Y-%m-%d %H:%M:%S')}")

        # Write batch as CSV (no header — schema defined on the reader side)
        filename = os.path.join(stream_dir, f"batch_{batch_num:05d}.csv")
        with open(filename, "w") as f:
            f.write("\n".join(rows))

        batch_num += 1
        stop_event.wait(interval)  # sleep, but wake up if stop is signaled

    print(f"Simulator stopped after {batch_num} batches.")


# Convert catalog track_ids to pandas for the simulator
catalog_pdf = catalog.select("track_id").toPandas()

# Create the stop event (we will use this to shut down the simulator)
stop_event = threading.Event()

print("Simulator function defined. We will start it when we need it.")

## Task 1: Read the Stream

Structured Streaming treats incoming data as an unbounded table that grows with each new batch. We use `spark.readStream` to define the source.

**Key rule:** You must define the schema explicitly — Spark cannot infer it from a stream (there may be no files yet when the query starts).

In [ ]:
# Define schema for play events (must match what the simulator writes)
play_schema = StructType([
    StructField("track_id", StringType(), True),
    StructField("user_id", IntegerType(), True),
    StructField("timestamp", TimestampType(), True)
])

# Read the stream
raw_stream = (
    spark.readStream
    .format("csv")
    .schema(play_schema)
    .option("header", "false")           # simulator writes no header
    .option("maxFilesPerTrigger", 2)     # process 2 files per micro-batch
    .load(STREAM_DIR)
)

print(f"Is streaming DF: {raw_stream.isStreaming}")
raw_stream.printSchema()

### Start the simulator and inspect with console sink

The **console sink** prints each micro-batch to stdout. We use `outputMode("append")` because there is no aggregation yet — each row appears once.

In [ ]:
# Start the simulator in a background thread
stop_event.clear()
sim_thread = threading.Thread(
    target=play_event_simulator,
    args=(catalog_pdf, STREAM_DIR, stop_event),
    kwargs={"batch_size": 75, "interval": 4},
    daemon=True
)
sim_thread.start()
print("Simulator started.")

In [ ]:
# Console sink — just peek at the raw stream
query_console = (
    raw_stream.writeStream
    .format("console")
    .outputMode("append")
    .option("truncate", False)
    .option("numRows", 10)
    .option("checkpointLocation", os.path.join(CHECKPOINT_BASE, "task1_console"))
    .queryName("task1_raw_peek")
    .start()
)

# Let it run for ~15 seconds so a few batches appear
query_console.awaitTermination(15)
query_console.stop()
print("Console query stopped.")

> **Observe:** Each micro-batch shows a small table of `(track_id, user_id, timestamp)`. The `Batch` number increments. Spark processes files as they appear — this is the fundamental Structured Streaming loop.

## Task 2: Stream-Static Join

Raw play events only carry `track_id`. To analyze genres and audio features, we join the stream against our static Spotify catalog. This is a **stream-static join** — one side is unbounded (stream), the other is a fixed table loaded into memory.

In [ ]:
# Select the columns we need from the catalog
catalog_slim = catalog.select(
    "track_id", "artists", "track_name", "track_genre",
    "energy", "valence", "danceability", "tempo", "popularity"
)

# Join streaming events with the static catalog
enriched_stream = raw_stream.join(catalog_slim, on="track_id", how="inner")

print("Enriched schema:")
enriched_stream.printSchema()

In [ ]:
# Peek at enriched events
query_enriched = (
    enriched_stream
    .select("timestamp", "user_id", "track_name", "artists", "track_genre", "energy", "valence")
    .writeStream
    .format("console")
    .outputMode("append")
    .option("truncate", True)
    .option("numRows", 10)
    .option("checkpointLocation", os.path.join(CHECKPOINT_BASE, "task2_enriched"))
    .queryName("task2_enriched_peek")
    .start()
)

query_enriched.awaitTermination(15)
query_enriched.stop()
print("Enriched query stopped.")

> **Observe:** Each row now has the full track metadata — genre, artist, audio features — even though the stream only carried `track_id`. This is the power of stream-static joins: lightweight events enriched by a reference table.

## Task 3: Running Counts (Complete Mode)

Count total plays per genre across *all* data seen so far. This is an **unbounded aggregation** — the counts never reset.

We use `outputMode("complete")` because every micro-batch outputs the *entire* updated result table, not just new rows.

In [ ]:
genre_counts = (
    enriched_stream
    .groupBy("track_genre")
    .count()
    .orderBy(F.desc("count"))
)

query_genre_counts = (
    genre_counts.writeStream
    .format("console")
    .outputMode("complete")       # complete = output the full table each batch
    .option("truncate", False)
    .option("numRows", 10)
    .option("checkpointLocation", os.path.join(CHECKPOINT_BASE, "task3_genre_counts"))
    .queryName("task3_genre_counts")
    .start()
)

query_genre_counts.awaitTermination(15)
query_genre_counts.stop()
print("Genre counts query stopped.")

> **Observe:** The counts grow with each batch. Every micro-batch re-outputs the *full* table (that is what `complete` mode means). Over time, genres converge toward uniform counts because the simulator samples uniformly. In a real system, popular genres would dominate.

## Task 4: Windowed Aggregation — Tumbling Windows

Running counts tell you *all-time* totals. But what is trending *right now*? Windowed aggregation groups events into fixed time buckets.

A **tumbling window** divides time into non-overlapping intervals. Here: 2-minute windows. Each event belongs to exactly one window.

In [ ]:
genre_windowed = (
    enriched_stream
    .groupBy(
        F.window("timestamp", "2 minutes"),   # tumbling window: 2 min, no slide
        "track_genre"
    )
    .count()
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "track_genre",
        "count"
    )
    .orderBy("window_start", F.desc("count"))
)

query_windowed = (
    genre_windowed.writeStream
    .format("console")
    .outputMode("complete")
    .option("truncate", False)
    .option("numRows", 30)
    .option("checkpointLocation", os.path.join(CHECKPOINT_BASE, "task4_tumbling"))
    .queryName("task4_tumbling_window")
    .start()
)

query_windowed.awaitTermination(15)
query_windowed.stop()
print("Tumbling window query stopped.")

> **Observe:** Each row now belongs to a specific 2-minute window. You can see *which genres are hot in which time bucket*. Compare counts across windows — the most recent window is still accumulating, while older windows are "closed" (no new events will fall into them).

**Key difference from Task 3:** Running counts grow forever. Windowed counts give you a *time-bounded* view — exactly what a real-time trending dashboard needs.

## Task 5: Sliding Window — Mood Tracker

A **sliding window** overlaps with its neighbors. Window size = 5 minutes, slide interval = 1 minute. Each event can belong to *multiple* windows.

Use case: "What is the average mood of our listeners *right now*?" We track two Spotify audio features:
- **energy** (0–1): intensity and activity level
- **valence** (0–1): musical positivity (happy vs. sad)

High energy + high valence = party mood. Low energy + low valence = chill/melancholy.

In [ ]:
mood_stream = (
    enriched_stream
    .groupBy(
        F.window("timestamp", "5 minutes", "1 minute")   # 5-min window, 1-min slide
    )
    .agg(
        F.round(F.avg("energy"), 3).alias("avg_energy"),
        F.round(F.avg("valence"), 3).alias("avg_valence"),
        F.round(F.avg("danceability"), 3).alias("avg_danceability"),
        F.count("*").alias("play_count")
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "avg_energy", "avg_valence", "avg_danceability", "play_count"
    )
    .orderBy("window_start")
)

query_mood = (
    mood_stream.writeStream
    .format("console")
    .outputMode("complete")
    .option("truncate", False)
    .option("numRows", 15)
    .option("checkpointLocation", os.path.join(CHECKPOINT_BASE, "task5_mood"))
    .queryName("task5_mood_tracker")
    .start()
)

query_mood.awaitTermination(15)
query_mood.stop()
print("Mood tracker query stopped.")

> **Observe:** Windows overlap — a single event appears in 5 windows. This gives a smoother, more responsive signal than tumbling windows.

**Question:** Why do the values stay roughly constant across windows? (Hint: the simulator samples uniformly from the catalog. In a real system, mood would shift as listeners change what they play during the day — upbeat in the morning, chill at night.)

In [ ]:
query_mood = (
    mood_stream.writeStream
    .format("memory")
    .outputMode("complete")
    .queryName("mood_table")
    .start()
)
query_mood.awaitTermination(30)

# Now query the in-memory table
mood_pdf = spark.sql("SELECT * FROM mood_table ORDER BY window_start").toPandas()
query_mood.stop()

# Plot
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.plot(mood_pdf["window_start"], mood_pdf["avg_energy"], marker="o")
plt.xlabel("Window Start")
plt.ylabel("Avg Energy")
plt.title("Average Energy Over Time (Sliding Window)")
plt.xticks(rotation=45)
plt.grid()
plt.ylim(0, None)
plt.tight_layout()
plt.show()


## Task 6: Watermarking — Handling Late Data

In real systems, events can arrive late — network delays, buffered mobile clients, retries. Our simulator deliberately injects events with timestamps 5–15 minutes in the past.

**Problem:** Without watermarking, Spark must keep *all* window state forever in case a late event arrives for an old window. This causes unbounded memory growth.

**Solution:** `withWatermark("timestamp", "10 minutes")` tells Spark: "If an event arrives more than 10 minutes late, drop it. After that threshold, finalize and discard old window state."

### Without watermark (for comparison)

In [ ]:
# Without watermark — all late events are included, state grows forever

no_wm_stream = (
    enriched_stream
    .groupBy(
        F.window("timestamp", "2 minutes"),
        "track_genre"
    )
    .count()
)

query_no_wm = (
    no_wm_stream
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "track_genre", "count"
    )
    .orderBy("window_start")
    .writeStream
    .format("console")
    .outputMode("complete")
    .option("truncate", False)
    .option("numRows", 10)
    .option("checkpointLocation", os.path.join(CHECKPOINT_BASE, "task6_no_wm"))
    .queryName("task6_no_watermark")
    .start()
)

query_no_wm.awaitTermination(15)
query_no_wm.stop()
print("No-watermark query stopped.")

### With watermark

In [ ]:
# With watermark — events more than 1 minute late are dropped
# State for windows older than (max_event_time - 1 minute) is cleaned up
wm_stream = (
    enriched_stream
    .withWatermark("timestamp", "1 minute")
    .groupBy(
        F.window("timestamp", "2 minutes"),
        "track_genre"
    )
    .count()
)

query_wm = (
    wm_stream
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "track_genre", "count"
    )
    .writeStream
    .format("console")
    .outputMode("append")
    .option("truncate", False)
    .option("numRows", 20)
    .option("checkpointLocation", os.path.join(CHECKPOINT_BASE, "task6_wm"))
    .queryName("task6_watermark")
    .start()
)

query_wm.awaitTermination(20)
query_wm.stop()
print("Watermark query stopped.")

> **Observe:** Compare the two outputs:
> - **Without watermark:** You may see windows stretching far into the past (from late events). Spark keeps *all* window state.
> - **With watermark:** Old windows are finalized and cleaned up. Late events beyond the threshold are silently dropped.
>
> In production, watermarking is essential. Without it, a 24/7 pipeline would eventually run out of memory.

**Key concept:** The watermark is not a wall clock timer. It advances based on the *maximum event time seen so far*. If the newest event has timestamp 12:30, Spark drops any event with timestamp before 12:20 (with a 10-minute watermark).

## Task 7: Monitoring a Streaming Query

In production, you need to monitor pipeline health: Is it keeping up? Are batches processing fast enough? Spark provides `query.lastProgress` and `query.status` for this.

In [ ]:
# Start a simple query to monitor
monitor_stream = (
    enriched_stream
    .groupBy(
        F.window("timestamp", "2 minutes"),
        "track_genre"
    )
    .count()
)

query_monitor = (
    monitor_stream
    .writeStream
    .format("console")
    .outputMode("complete")
    .option("truncate", True)
    .option("numRows", 5)
    .option("checkpointLocation", os.path.join(CHECKPOINT_BASE, "task7_monitor"))
    .queryName("task7_monitor")
    .start()
)

# Let it process a few batches
time.sleep(6)

In [ ]:
# Check if the query is still running
print(f"Is active: {query_monitor.isActive}")
print(f"Query name: {query_monitor.name}")
print(f"Query ID: {query_monitor.id}")
print()

# Last progress report
progress = query_monitor.lastProgress
if progress:
    print("=== Last Progress ===")
    print(f"  Batch ID:               {progress.get('batchId', 'N/A')}")
    print(f"  Input rows/sec:         {progress.get('inputRowsPerSecond', 'N/A')}")
    print(f"  Processed rows/sec:     {progress.get('processedRowsPerSecond', 'N/A')}")
    print(f"  Batch duration (ms):    {progress.get('batchDuration', 'N/A')}")
    # print(f"  Batch duration (ms):    {progress.get('durationMs', {}).get('triggerExecution', 'N/A')}")
    print(f"  Num input rows:         {progress.get('numInputRows', 'N/A')}")

    # Sources info
    sources = progress.get("sources", [])
    for src in sources:
        print(f"  Source description:      {src.get('description', 'N/A')}")
        print(f"  Start offset:            {src.get('startOffset', 'N/A')}")
        print(f"  End offset:              {src.get('endOffset', 'N/A')}")
else:
    print("No progress report yet — query may still be initializing.")

print()

# Current status
status = query_monitor.status
print("=== Current Status ===")
for key, value in status.items():
    print(f"  {key}: {value}")

> **Observe:** Two critical metrics:
> - **inputRowsPerSecond** — how fast data is arriving
> - **processedRowsPerSecond** — how fast Spark is consuming it
>
> If `processedRowsPerSecond` consistently exceeds `inputRowsPerSecond`, the pipeline is healthy. If not, you have a backlog problem — scale up, optimize joins, or reduce window complexity.

In [ ]:
# List ALL active streaming queries in this session
print("Active streaming queries:")
for q in spark.streams.active:
    print(f"  {q.name} (id={q.id}, active={q.isActive})")

query_monitor.stop()
print("\nMonitor query stopped.")

## Task 8: Stop All Streams — Clean Shutdown

Always stop your streaming queries and the simulator before ending the session. Leaving queries running consumes resources and locks checkpoint directories.

In [ ]:
# Stop the simulator
stop_event.set()
print("Simulator stop signal sent.")

# Give the thread a moment to finish
time.sleep(2)

# Stop ALL active streaming queries
active = spark.streams.active
print(f"\nStopping {len(active)} active queries...")
for q in active:
    print(f"  Stopping: {q.name}")
    q.stop()

print("\nAll queries stopped.")

In [ ]:
# Verify nothing is running
assert len(spark.streams.active) == 0, "Some queries are still running!"
print("All streaming queries confirmed stopped.")
print(f"Simulator thread alive: {sim_thread.is_alive()}")

### Cleanup (optional)

Remove the stream data and checkpoint directories. Run this only if you are done and want a fresh start.

In [ ]:
# Uncomment to clean up:
# clean_dirs()
# print("Cleaned up stream data and checkpoints.")

## Discussion Questions

Answer these before leaving:

1. **Append vs. Complete vs. Update:** When would you use each output mode? Which mode did we use for aggregations, and why?

2. **Tumbling vs. Sliding windows:** A dashboard shows "plays per genre in the last 5 minutes, updated every minute." Is this a tumbling or sliding window? What are the window size and slide interval?

3. **Watermarking trade-off:** If you set the watermark threshold to 1 minute instead of 10 minutes, what happens to (a) memory usage, and (b) data completeness?

4. **Stream-static join:** Why can we use an inner join here without worrying about output mode restrictions? What would change if both sides were streams?

5. **Production:** Name two things you would add to this pipeline before deploying it to production (think: fault tolerance, output destination, alerting).

In [ ]:
spark.stop()
print("SparkSession stopped. Lab complete.")